In [2]:
import math 
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [6]:
tfm = transforms.Compose([
    transforms.ToTensor(),
     transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

print("train examples:", len(train_ds))
print("test  examples:", len(test_ds))

# peek at ONE example to see the raw shape before we flatten it
img, label = train_ds[0]
print("one image tensor shape:", img.shape, "| label:", label)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:08<00:00, 1140925.45it/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 115440.00it/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:01<00:00, 1181838.20it/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 63328.03it/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

train examples: 60000
test  examples: 10000
one image tensor shape: torch.Size([1, 28, 28]) | label: 5


In [7]:
class FFN(nn.Module):
    def __init__(self, d_in=784, d_hidden=1024, d_out=10):
        super().__init__()
        self.fc1 = nn.Linear(d_in,     d_hidden)   # 784  -> 1024
        self.fc2 = nn.Linear(d_hidden, d_hidden)   # 1024 -> 1024  (your 1024x1024 tensor)
        self.fc3 = nn.Linear(d_hidden, d_out)      # 1024 -> 10

    def forward(self, x):
        x = x.view(x.size(0), -1)      # (B, 1, 28, 28) -> (B, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)             # logits (B, 10)

model = FFN().to(device)
print(model)

FFN(
  (fc1): Linear(in_features=784, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=1024, bias=True)
  (fc3): Linear(in_features=1024, out_features=10, bias=True)
)


In [28]:
# (a) The quickest look — grab one weight matrix directly by name
print("fc1.weight shape:", model.fc1.weight.shape)
print(model.fc1.weight)          # the actual (1024, 784) random matrix
print()
print("fc1.bias shape:", model.fc1.bias.shape)
print(model.fc1.bias)            # the (1024,) bias vector

fc1.weight shape: torch.Size([1024, 784])
Parameter containing:
tensor([[-0.0049, -0.0101, -0.0096,  ..., -0.0338,  0.0035,  0.0210],
        [-0.0039, -0.0282,  0.0243,  ...,  0.0247,  0.0318, -0.0053],
        [ 0.0184, -0.0088,  0.0099,  ..., -0.0292, -0.0100,  0.0256],
        ...,
        [-0.0118,  0.0171, -0.0117,  ..., -0.0349,  0.0062, -0.0273],
        [ 0.0084,  0.0299, -0.0088,  ...,  0.0220, -0.0095, -0.0164],
        [-0.0196, -0.0204, -0.0110,  ..., -0.0282, -0.0053, -0.0027]],
       device='mps:0', requires_grad=True)

fc1.bias shape: torch.Size([1024])
Parameter containing:
tensor([ 0.0103,  0.0298, -0.0291,  ...,  0.0083, -0.0078, -0.0205],
       device='mps:0', requires_grad=True)


In [31]:
for name, p in model.named_parameters():
    print(name, p.shape)

fc1.weight torch.Size([1024, 784])
fc1.bias torch.Size([1024])
fc2.weight torch.Size([1024, 1024])
fc2.bias torch.Size([1024])
fc3.weight torch.Size([10, 1024])
fc3.bias torch.Size([10])


In [29]:
# ─── Cell 5: enumerate ALL parameters (this is Topic 1's real anatomy) ────
print(f"{'name':<12}{'shape':<18}{'numel':>10}")
print("-" * 40)

total = 0
for name, p in model.named_parameters():
    n = p.numel()
    total += n
    print(f"{name:<12}{str(tuple(p.shape)):<18}{n:>10,}")

print("-" * 40)
print(f"{'TOTAL P':<12}{'':<18}{total:>10,}")


name        shape                  numel
----------------------------------------
fc1.weight  (1024, 784)          802,816
fc1.bias    (1024,)                1,024
fc2.weight  (1024, 1024)       1,048,576
fc2.bias    (1024,)                1,024
fc3.weight  (10, 1024)            10,240
fc3.bias    (10,)                     10
----------------------------------------
TOTAL P                        1,863,690


In [40]:
flats= []
offset_table =[]
cursor = 0

for name, p in model.named_parameters():
    flat = p.detach().reshape(-1)
    n = flat.numel()
    offset_table.append({
        "name": name,
        "shape": tuple(p.shape),
        "start": cursor,
        "end": cursor+n
    })
    flats.append(flat)
    cursor += n

ribbon = torch.cat(flats)

In [42]:
ribbon.shape

torch.Size([1863690])

In [35]:
print("ribbon shape:", ribbon.shape)
print("ribbon dtype:", ribbon.dtype)
print("ribbon.numel() == P ?", ribbon.numel() == 1_863_690)

# ─── Cell 7: inspect the offset table (the metadata map) ──────────────────
print(f"{'name':<12}{'shape':<16}{'start':>10}{'end':>12}{'len':>10}")
print("-" * 60)
for e in offset_table:
    print(f"{e['name']:<12}{str(e['shape']):<16}{e['start']:>10,}{e['end']:>12,}{e['end']-e['start']:>10,}")


ribbon shape: torch.Size([1863690])
ribbon dtype: torch.float32
ribbon.numel() == P ? True
name        shape                start         end       len
------------------------------------------------------------
fc1.weight  (1024, 784)              0     802,816   802,816
fc1.bias    (1024,)            802,816     803,840     1,024
fc2.weight  (1024, 1024)       803,840   1,852,416 1,048,576
fc2.bias    (1024,)          1,852,416   1,853,440     1,024
fc3.weight  (10, 1024)       1,853,440   1,863,680    10,240
fc3.bias    (10,)            1,863,680   1,863,690        10


In [43]:
N = 8                                    # world size (simulated ranks)

P   = ribbon.numel()                     # 1,863,690  — true param count
pad = (-P) % N                           # how many zeros to append
L   = P + pad                            # padded ribbon length
shard_len = L // N                       # length of each rank's shard

print(f"P (real params)     : {P:,}")
print(f"N (world size)      : {N}")
print(f"P % N               : {P % N}   <- nonzero => needs padding")
print(f"pad (zeros to add)  : {pad}")
print(f"L (padded length)   : {L:,}")
print(f"L % N               : {L % N}   <- must be 0")
print(f"shard_len (L / N)   : {shard_len:,}")

P (real params)     : 1,863,690
N (world size)      : 8
P % N               : 2   <- nonzero => needs padding
pad (zeros to add)  : 6
L (padded length)   : 1,863,696
L % N               : 0   <- must be 0
shard_len (L / N)   : 232,962


In [46]:
import torch.nn.functional as F

ribbon_padded = F.pad(ribbon, (0, pad))  # append `pad` zeros on the right

print("ribbon        :", ribbon.shape)
print("ribbon_padded :", ribbon_padded.shape)
print("last 8 values of padded ribbon:", ribbon_padded[-8:])
print("divisible now :", ribbon_padded.numel() % N == 0)

ribbon        : torch.Size([1863690])
ribbon_padded : torch.Size([1863696])
last 8 values of padded ribbon: tensor([0.0149, 0.0140, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       device='mps:0')
divisible now : True


In [47]:
shards = list(ribbon_padded.split(shard_len))   # N pieces, each shard_len long

print(f"number of shards      : {len(shards)}")
print(f"each shard length     : {shard_len:,}")
print()
for i, s in enumerate(shards):
    print(f"  rank {i}: shard shape {tuple(s.shape)}  ->  holds {s.numel():,} numbers")

number of shards      : 8
each shard length     : 232,962

  rank 0: shard shape (232962,)  ->  holds 232,962 numbers
  rank 1: shard shape (232962,)  ->  holds 232,962 numbers
  rank 2: shard shape (232962,)  ->  holds 232,962 numbers
  rank 3: shard shape (232962,)  ->  holds 232,962 numbers
  rank 4: shard shape (232962,)  ->  holds 232,962 numbers
  rank 5: shard shape (232962,)  ->  holds 232,962 numbers
  rank 6: shard shape (232962,)  ->  holds 232,962 numbers
  rank 7: shard shape (232962,)  ->  holds 232,962 numbers


In [48]:
total_in_shards = sum(s.numel() for s in shards)

print("sum of all shard lengths :", f"{total_in_shards:,}")
print("padded ribbon length  L  :", f"{ribbon_padded.numel():,}")
print("match?                   :", total_in_shards == ribbon_padded.numel())
print()
# the crucial FSDP fact: each rank holds only 1/N of the model
print(f"each rank holds {shard_len:,} of {ribbon_padded.numel():,} numbers")
print(f"           = {shard_len / ribbon_padded.numel() * 100:.1f}%  (i.e. 1/{N})")

sum of all shard lengths : 1,863,696
padded ribbon length  L  : 1,863,696
match?                   : True

each rank holds 232,962 of 1,863,696 numbers
           = 12.5%  (i.e. 1/8)


In [49]:
# rebuilds the exact padded ribbon, byte-for-byte.
reassembled = torch.cat(shards)
print("reassembled == ribbon_padded ?", torch.equal(reassembled, ribbon_padded))

reassembled == ribbon_padded ? True


In [51]:
# Look up fc2.weight's span in the offset table
entry = next(e for e in offset_table if e["name"] == "fc2.weight")
start, end, shape = entry["start"], entry["end"], entry["shape"]
print(f"fc2.weight: ribbon positions [{start:,} : {end:,}]  original shape {shape}")

fc2.weight: ribbon positions [803,840 : 1,852,416]  original shape (1024, 1024)


In [52]:
# shard boundaries: shard k covers [k*shard_len : (k+1)*shard_len]
print("\nshard boundaries:")
for k in range(N):
    lo, hi = k * shard_len, (k + 1) * shard_len
    overlaps = (start < hi) and (end > lo)          # does fc2.weight touch this shard?
    mark = "  <-- fc2.weight lives here" if overlaps else ""
    print(f"  rank {k}: [{lo:>10,} : {hi:>10,}]{mark}")


shard boundaries:
  rank 0: [         0 :    232,962]
  rank 1: [   232,962 :    465,924]
  rank 2: [   465,924 :    698,886]
  rank 3: [   698,886 :    931,848]  <-- fc2.weight lives here
  rank 4: [   931,848 :  1,164,810]  <-- fc2.weight lives here
  rank 5: [ 1,164,810 :  1,397,772]  <-- fc2.weight lives here
  rank 6: [ 1,397,772 :  1,630,734]  <-- fc2.weight lives here
  rank 7: [ 1,630,734 :  1,863,696]  <-- fc2.weight lives here


In [53]:
# ─── Cell 14: rebuild fc2.weight from the shards, by hand ────────────────
# Step 1: glue the shards back into the full ribbon (this IS all-gather)
full_ribbon = torch.cat(shards)                 # touches all shards = "the network"

# Step 2: slice out fc2.weight's flat span using the offset table
flat_slice = full_ribbon[start:end]             # 1-D, length = end - start
print("flat_slice shape :", tuple(flat_slice.shape), " (still 1-D)")

# Step 3: reshape that flat run back to the original 2-D matrix
rebuilt = flat_slice.view(shape)                # (1024, 1024)
print("rebuilt shape    :", tuple(rebuilt.shape))

# ─── Cell 15: prove it matches the original parameter ────────────────────
original = model.fc2.weight.detach()
print("shapes match  :", rebuilt.shape == original.shape)
print("values match  :", torch.equal(rebuilt, original))
print("max abs diff  :", (rebuilt - original).abs().max().item())

flat_slice shape : (1048576,)  (still 1-D)
rebuilt shape    : (1024, 1024)
shapes match  : True
values match  : True
max abs diff  : 0.0


In [54]:
def all_gather(shards):
    """Collective: glue all N shards into the full padded ribbon.
    On real hardware this is a NCCL network transfer; here it's torch.cat.
    This is the ONLY place we touch all shards at once (= communication)."""
    return torch.cat(shards)                      # (L,) full ribbon

def unflatten(full_ribbon, offset_table):
    """Rebuild every named tensor from the ribbon using the offset table.
    Returns a dict: name -> tensor at its original shape."""
    params = {}
    for e in offset_table:
        flat_slice = full_ribbon[e["start"]:e["end"]]   # slice its span
        params[e["name"]] = flat_slice.view(e["shape"]) # reshape to original
    return params

In [ ]:
full_ribbon = all_gather(shards)                  # step 1: the collective
params = unflatten(full_ribbon, offset_table)     # steps 2+3, for ALL tensors

print("gathered full ribbon:", tuple(full_ribbon.shape))
print()
print(f"{'name':<12}{'rebuilt shape':<16}{'matches original?'}")
print("-" * 45)
for name, t in params.items():
    original = dict(model.named_parameters())[name].detach()
    print(f"{name:<12}{str(tuple(t.shape)):<16}{torch.equal(t, original)}")


gathered full ribbon: (1863696,)

name        rebuilt shape   matches original?
---------------------------------------------
fc1.weight  (1024, 784)     True
fc1.bias    (1024,)         True
fc2.weight  (1024, 1024)    True
fc2.bias    (1024,)         True
fc3.weight  (10, 1024)      True
fc3.bias    (10,)           True


In [56]:
def ffn_forward(params, x):
    """Run the FFN using weights from the gathered params dict.
    params: {"fc1.weight": ..., "fc1.bias": ..., ...} at original shapes.
    x: (B, 784) input batch."""
    x = F.linear(x, params["fc1.weight"], params["fc1.bias"]);  x = F.relu(x)
    x = F.linear(x, params["fc2.weight"], params["fc2.bias"]);  x = F.relu(x)
    x = F.linear(x, params["fc3.weight"], params["fc3.bias"])
    return x                                    # logits (B, 10)


In [57]:
loader = DataLoader(train_ds, batch_size=N * 32, shuffle=True, drop_last=True)
xb, yb = next(iter(loader))                     # (256, 1, 28, 28), (256,)

xb = xb.view(xb.size(0), -1).to(device)         # flatten images -> (256, 784)
yb = yb.to(device)
micro = xb.size(0) // N                          # 32 examples per rank

print("full batch     :", tuple(xb.shape), tuple(yb.shape))
print("per-rank micro :", micro, "examples")
print("=> rank i sees  xb[i*32:(i+1)*32], its OWN slice of data")

full batch     : (256, 784) (256,)
per-rank micro : 32 examples
=> rank i sees  xb[i*32:(i+1)*32], its OWN slice of data


In [58]:
losses = []
for i in range(N):
    # ---- ALL-GATHER: reconstruct full weights (transient) ----
    full_ribbon = all_gather(shards)
    params = unflatten(full_ribbon, offset_table)

    # ---- COMPUTE: this rank's OWN micro-batch ----
    xr = xb[i * micro:(i + 1) * micro]           # (32, 784)  <- different per rank
    yr = yb[i * micro:(i + 1) * micro]           # (32,)
    logits = ffn_forward(params, xr)             # (32, 10)
    loss = F.cross_entropy(logits, yr)
    losses.append(loss.item())

    # ---- DISCARD: drop the full ribbon, fall back to 1/N shards ----
    del full_ribbon, params                      # the borrowed (N-1)/N is gone

    print(f"rank {i}: xr {tuple(xr.shape)} -> logits {tuple(logits.shape)} -> loss {loss.item():.4f}")

print(f"\nmean loss across ranks: {sum(losses)/N:.4f}")


rank 0: xr (32, 784) -> logits (32, 10) -> loss 2.2773
rank 1: xr (32, 784) -> logits (32, 10) -> loss 2.3115
rank 2: xr (32, 784) -> logits (32, 10) -> loss 2.2887
rank 3: xr (32, 784) -> logits (32, 10) -> loss 2.2734
rank 4: xr (32, 784) -> logits (32, 10) -> loss 2.3055
rank 5: xr (32, 784) -> logits (32, 10) -> loss 2.3150
rank 6: xr (32, 784) -> logits (32, 10) -> loss 2.2973
rank 7: xr (32, 784) -> logits (32, 10) -> loss 2.3046

mean loss across ranks: 2.2967
